# Walmart Sales Exploratory Data Analysis (EDA)

This notebook performs an in-depth exploratory data analysis of the 5-year Walmart sales transactions dataset (2019-2023). We analyze sales trends, category performance, margins, and customer rating dynamics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

## 1. Load Cleaned Dataset

In [ ]:
df = pd.read_csv('walmart_cleaned.csv')
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
df.head()

## 2. Sales Over Time (Temporal Trends)

Let's look at the growth of sales over the years and check for any monthly seasonality patterns.

In [ ]:
# Aggregate sales by month
monthly_sales = df.resample('M', on='date')['total'].sum().reset_index()

plt.figure(figsize=(14, 6))
plt.plot(monthly_sales['date'], monthly_sales['total'], marker='o', color='royalblue', linewidth=2.5)
plt.title('Monthly Sales Trend (2019 - 2023)', fontsize=16, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Total Sales ($)')
plt.tight_layout()
plt.savefig('monthly_sales_trend.png', dpi=300)
plt.show()

We can also aggregate by day of the week to see which days are busiest across all branches.

In [ ]:
df['day_name'] = df['date'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly_pattern = df.groupby('day_name')['total'].sum().reindex(day_order).reset_index()

sns.barplot(data=weekly_pattern, x='day_name', y='total', hue='day_name', legend=False, palette='viridis')
plt.title('Sales Revenue by Day of the Week', fontsize=16, fontweight='bold')
plt.xlabel('Day of the Week')
plt.ylabel('Total Sales ($)')
plt.show()

## 3. Product Category Analysis

Let's see which categories generate the most revenue and check their average profit margins.

In [ ]:
category_sales = df.groupby('category').agg(
    total_revenue=('total', 'sum'),
    avg_profit_margin=('profit_margin', 'mean')
).reset_index().sort_values(by='total_revenue', ascending=False)

fig, ax1 = plt.subplots(figsize=(14, 6))

# Primary axis for revenue
sns.barplot(data=category_sales, x='category', y='total_revenue', ax=ax1, palette='Blues_r', hue='category', legend=False)
ax1.set_title('Category Revenue vs. Average Profit Margin', fontsize=16, fontweight='bold')
ax1.set_ylabel('Total Revenue ($)', color='blue')
ax1.set_xlabel('Category')
ax1.tick_params(axis='y', labelcolor='blue')

# Secondary axis for profit margin
ax2 = ax1.twinx()
sns.lineplot(data=category_sales, x='category', y='avg_profit_margin', color='orange', marker='s', markersize=8, linewidth=2.5, ax=ax2, sort=False)
ax2.set_ylabel('Average Profit Margin', color='darkorange')
ax2.tick_params(axis='y', labelcolor='darkorange')
ax2.grid(False)

plt.tight_layout()
plt.savefig('category_performance.png', dpi=300)
plt.show()

## 4. Rating and Customer Satisfaction Analysis

Is there a correlation between customer ratings and product category or payment methods?

In [ ]:
sns.boxplot(data=df, x='payment_method', y='rating', hue='payment_method', palette='Set2', legend=False)
plt.title('Customer Rating Distribution by Payment Method', fontsize=16, fontweight='bold')
plt.xlabel('Payment Method')
plt.ylabel('Customer Rating')
plt.show()

## 5. Feature Correlation Heatmap

In [ ]:
numerical_cols = ['unit_price', 'quantity', 'rating', 'profit_margin', 'total']
corr_matrix = df[numerical_cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.3f', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Numerical Features Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300)
plt.show()